# 🧬 Genomic Embryo Selection with Transformer + GNN on A100

## Complete Pipeline: Data → Training → Results

**Features:**
- ✅ Automatic repository download
- ✅ Real AlphaFold structures + KEGG pathways + STRING networks
- ✅ Genomic Transformer with biological attention
- ✅ Multi-task learning (3+ traits)
- ✅ Structural priors from AlphaFold
- ✅ Mixed precision training (FP16/BF16)
- ✅ Optimized for NVIDIA A100 (40GB)
- ✅ TensorBoard visualization
- ✅ Automatic checkpointing

**Runtime:** Use **GPU (A100)** runtime

**Expected time:** ~2 hours for full pipeline

## 0. Setup: Check GPU and Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Check for bfloat16 support (A100)
    if torch.cuda.is_bf16_supported():
        print("✓ BFloat16 supported (A100 detected!)")
    else:
        print("⚠ BFloat16 not supported (not A100)")

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install -q biopython requests tqdm pandas numpy scipy scikit-learn
!pip install -q tensorboard wandb
!pip install -q pybind11 setuptools

print("✓ Dependencies installed")

## 1. Clone Repository and Setup

In [ ]:
# Clone repository
!git clone https://github.com/tomm03o/idao.git
%cd idao/genomic-selection

# Verify structure
!ls -lh
print("\n✓ Repository cloned")

In [ ]:
# Add src to Python path
import sys
from pathlib import Path

project_root = Path.cwd()
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path[:3]}")

## 2. Download Real Biological Data

**Data sources:**
- AlphaFold structures (β-Casein, β-Lactoglobulin)
- KEGG pathways (20 growth/metabolism pathways)
- STRING protein networks (5.2M interactions)

In [ ]:
# Download AlphaFold structures
print("Downloading AlphaFold structures...")
!python tests/test_real_alphafold.py 2>&1 | grep -E "(SUCCESS|Downloaded|Found)"

# Verify downloads
!ls -lh data/structures/
print("\n✓ AlphaFold structures downloaded")

In [ ]:
# Download KEGG pathways
print("Downloading KEGG pathways...")
!python data/downloaders/download_kegg.py

# Verify
!ls -lh data/raw/kegg/
print("\n✓ KEGG pathways downloaded")

In [ ]:
# Download STRING networks (this may take 5-10 minutes)
print("Downloading STRING protein networks...")
print("This will download ~461 MB of data")

!python data/downloaders/download_string.py

# Verify
!ls -lh data/raw/string/
print("\n✓ STRING networks downloaded")

## 3. Verify Data Integration

In [ ]:
# Run demo to verify all data is integrated
!python demo_real_data.py

print("\n✓ Data verification complete")

## 4. Create Simulated Genotype Data

For this demo, we'll use simulated genotypes.
In production, replace with real cattle/dog/horse genotypes.

In [ ]:
import numpy as np
import torch
from pathlib import Path

# Simulation parameters
N_SAMPLES = 2000  # Number of animals (embryos)
N_SNPS = 50000    # Number of SNPs (adjust based on GPU memory)
N_TASKS = 3       # Number of traits (milk yield, protein %, fat %)
H2 = 0.4          # Heritability
P_CAUSAL = 0.01   # Proportion of causal SNPs

print(f"Simulating genomic data:")
print(f"  Samples: {N_SAMPLES:,}")
print(f"  SNPs: {N_SNPS:,}")
print(f"  Tasks: {N_TASKS}")
print(f"  Heritability: {H2}")

# Simulate genotypes {0, 1, 2}
np.random.seed(42)
genotypes = np.random.randint(0, 3, size=(N_SAMPLES, N_SNPS), dtype=np.int64)

# Allele frequencies
allele_freq = np.random.beta(2, 5, size=N_SNPS).astype(np.float32)

# Functional annotations (10 classes)
functional_class = np.random.randint(0, 10, size=N_SNPS, dtype=np.int64)

# LD scores
ld_score = np.random.exponential(10, size=N_SNPS).astype(np.float32)

# Structural weights from reverse mapping
# Most SNPs = 1.0, critical protein domains = 100.0
structural_weight = np.ones(N_SNPS, dtype=np.float32)
n_priority = int(N_SNPS * 0.001)  # 0.1% high-priority SNPs
priority_indices = np.random.choice(N_SNPS, n_priority, replace=False)
structural_weight[priority_indices] = 100.0

print(f"  High-priority SNPs (100× weight): {n_priority:,} ({100*n_priority/N_SNPS:.2f}%)")

# Distance to nearest gene
distance_to_gene = np.random.exponential(5000, size=N_SNPS).astype(np.float32)

# Genomic positions (sorted)
positions = np.sort(np.random.randint(1, 250_000_000, size=N_SNPS)).astype(np.int64)

# Simulate phenotypes for multiple tasks
task_names = ['milk_yield', 'protein_pct', 'fat_pct']
targets = {}

for task_idx, task_name in enumerate(task_names):
    # True causal effects (sparse)
    n_causal = int(N_SNPS * P_CAUSAL)
    true_beta = np.zeros(N_SNPS)
    
    # Prioritize structural high-weight SNPs as causal
    high_weight_mask = structural_weight > 1.0
    n_structural_causal = int(n_causal * 0.5)  # 50% from high-priority
    
    structural_causal = np.random.choice(
        np.where(high_weight_mask)[0],
        size=min(n_structural_causal, high_weight_mask.sum()),
        replace=False
    )
    
    random_causal = np.random.choice(
        np.where(~high_weight_mask)[0],
        size=n_causal - len(structural_causal),
        replace=False
    )
    
    causal_indices = np.concatenate([structural_causal, random_causal])
    true_beta[causal_indices] = np.random.randn(len(causal_indices)) * 0.1
    
    # Genetic component
    genetic = (genotypes * true_beta).sum(axis=1)
    genetic_std = genetic.std()
    
    # Environmental noise
    noise_std = genetic_std * np.sqrt((1 - H2) / H2)
    noise = np.random.randn(N_SAMPLES) * noise_std
    
    # Phenotypes
    phenotypes = genetic + noise
    
    # Standardize
    phenotypes = (phenotypes - phenotypes.mean()) / phenotypes.std()
    
    targets[task_name] = phenotypes.astype(np.float32)
    
    print(f"\n  Task '{task_name}':")
    print(f"    Causal SNPs: {len(causal_indices):,}")
    print(f"    From structural priors: {len(structural_causal):,}")
    print(f"    Phenotype std: {phenotypes.std():.3f}")

# Save to disk
data_dir = Path("data/simulated")
data_dir.mkdir(exist_ok=True, parents=True)

np.savez_compressed(
    data_dir / "genomic_data.npz",
    genotypes=genotypes,
    allele_freq=allele_freq,
    functional_class=functional_class,
    ld_score=ld_score,
    structural_weight=structural_weight,
    distance_to_gene=distance_to_gene,
    positions=positions,
    **targets
)

print(f"\n✓ Data saved to {data_dir / 'genomic_data.npz'}")
print(f"  File size: {(data_dir / 'genomic_data.npz').stat().st_size / 1e6:.1f} MB")

## 5. Create Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GenomicDataset(Dataset):
    """Dataset for genomic data."""
    
    def __init__(self, data_path, train=True, train_split=0.8):
        # Load data
        data = np.load(data_path)
        
        n_samples = data['genotypes'].shape[0]
        n_train = int(n_samples * train_split)
        
        if train:
            indices = slice(0, n_train)
        else:
            indices = slice(n_train, n_samples)
        
        self.genotypes = data['genotypes'][indices]
        self.allele_freq = data['allele_freq']
        self.functional_class = data['functional_class']
        self.ld_score = data['ld_score']
        self.structural_weight = data['structural_weight']
        self.distance_to_gene = data['distance_to_gene']
        self.positions = data['positions']
        
        self.targets = {
            task: data[task][indices]
            for task in task_names
        }
        
        self.n_samples = self.genotypes.shape[0]
        self.n_snps = self.genotypes.shape[1]
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        return {
            'genotypes': torch.from_numpy(self.genotypes[idx]).long(),
            'allele_freq': torch.from_numpy(self.allele_freq).float(),
            'functional_class': torch.from_numpy(self.functional_class).long(),
            'ld_score': torch.from_numpy(self.ld_score).float(),
            'structural_weight': torch.from_numpy(self.structural_weight).float(),
            'distance_to_gene': torch.from_numpy(self.distance_to_gene).float(),
            'positions': torch.from_numpy(self.positions).long(),
            'targets': {
                task: torch.tensor([vals[idx]], dtype=torch.float32)
                for task, vals in self.targets.items()
            }
        }

# Create datasets
train_dataset = GenomicDataset('data/simulated/genomic_data.npz', train=True)
val_dataset = GenomicDataset('data/simulated/genomic_data.npz', train=False)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")
print(f"SNPs per sample: {train_dataset.n_snps:,}")

# Test loading one batch
sample = train_dataset[0]
print(f"\nSample shapes:")
for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v.shape}")

print("\n✓ Datasets created")

## 6. Create Genomic Transformer Model

In [ ]:
# Import model
from models.genomic_transformer import GenomicTransformer

# Model configuration
model_config = {
    'd_model': 256,
    'n_heads': 8,
    'n_layers': 6,
    'dim_feedforward': 1024,
    'n_functional_classes': 10,
    'n_tasks': len(task_names),
    'dropout': 0.1,
    'use_checkpointing': True  # Memory efficient for A100
}

# Create model
model = GenomicTransformer(**model_config)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created:")
print(f"  Total parameters: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Trainable parameters: {n_trainable:,}")
print(f"  Device: {device}")
print(f"  Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\n✓ Model ready for training")

## 7. Setup Training Configuration

In [ ]:
from models.multi_task_trainer import TrainingConfig, MultiTaskTrainer

# Training configuration
config = TrainingConfig(
    # Model (matches model_config)
    d_model=256,
    n_heads=8,
    n_layers=6,
    dim_feedforward=1024,
    
    # Training
    batch_size=16,  # Adjust based on GPU memory
    n_epochs=50,
    learning_rate=1e-4,
    weight_decay=1e-5,
    warmup_steps=500,
    gradient_clip=1.0,
    accumulation_steps=2,
    
    # Mixed precision (A100 optimized)
    use_amp=True,
    
    # Tasks
    task_names=task_names,
    task_weights={task: 1.0 for task in task_names},
    
    # Regularization
    dropout=0.1,
    
    # Checkpointing
    checkpoint_dir=Path("checkpoints/genomic_transformer"),
    save_every=5,
    
    # Early stopping
    patience=10,
    min_delta=1e-4,
    
    # Device
    device='cuda'
)

print("Training Configuration:")
print(f"  Batch size: {config.batch_size}")
print(f"  Epochs: {config.n_epochs}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Mixed precision: {config.use_amp}")
print(f"  Tasks: {config.task_names}")
print(f"  Checkpoint dir: {config.checkpoint_dir}")

## 8. Train Model

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Start TensorBoard
%tensorboard --logdir runs/

In [ ]:
# Create trainer
trainer = MultiTaskTrainer(
    model=model,
    config=config,
    train_dataset=train_dataset,
    val_dataset=val_dataset
)

print("✓ Trainer initialized")
print(f"  Batches per epoch: {len(trainer.train_loader)}")
print(f"  Total training steps: {len(trainer.train_loader) * config.n_epochs}")
print("\nStarting training...\n")

In [ ]:
# Train!
history = trainer.train()

print("\n" + "=" * 80)
print("✓ TRAINING COMPLETE!")
print("=" * 80)

## 9. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Progress', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Log scale
axes[1].plot(history['train_loss'], label='Train', linewidth=2)
axes[1].plot(history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss (log scale)', fontsize=12)
axes[1].set_title('Training Progress (Log Scale)', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best validation loss: {min(history['val_loss']):.4f}")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

## 10. Evaluate Model on Test Set

In [ ]:
# Load best model
best_checkpoint = torch.load(config.checkpoint_dir / "best_model.pt")
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print("✓ Loaded best model")
print(f"  Epoch: {best_checkpoint['epoch']}")
print(f"  Best val loss: {best_checkpoint['best_val_loss']:.4f}")

In [ ]:
# Evaluate on validation set
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_squared_error

predictions = {task: [] for task in task_names}
true_values = {task: [] for task in task_names}
uncertainties = {task: [] for task in task_names}

with torch.no_grad():
    for batch in tqdm(trainer.val_loader, desc="Evaluating"):
        # Move to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}
        
        # Predict
        preds, uncert = model(
            genotypes=batch['genotypes'],
            allele_freq=batch['allele_freq'],
            functional_class=batch['functional_class'],
            ld_score=batch['ld_score'],
            structural_weight=batch['structural_weight'],
            distance_to_gene=batch['distance_to_gene'],
            positions=batch['positions']
        )
        
        # Store predictions
        for task in task_names:
            predictions[task].extend(preds[f'task_{task_names.index(task)}'].cpu().numpy())
            true_values[task].extend(batch['targets'][task].cpu().numpy())
            uncertainties[task].extend(uncert[f'task_{task_names.index(task)}'].cpu().numpy())

# Convert to arrays
predictions = {k: np.array(v).flatten() for k, v in predictions.items()}
true_values = {k: np.array(v).flatten() for k, v in true_values.items()}
uncertainties = {k: np.array(v).flatten() for k, v in uncertainties.items()}

# Compute metrics
print("\n" + "=" * 80)
print("EVALUATION METRICS")
print("=" * 80)

for task in task_names:
    pred = predictions[task]
    true = true_values[task]
    
    r2 = r2_score(true, pred)
    corr, pval = pearsonr(true, pred)
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    
    print(f"\nTask: {task}")
    print(f"  R²: {r2:.4f}")
    print(f"  Correlation: {corr:.4f} (p={pval:.2e})")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  Mean uncertainty: {uncertainties[task].mean():.4f}")

In [ ]:
# Plot predictions vs true values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, task in enumerate(task_names):
    ax = axes[idx]
    
    pred = predictions[task]
    true = true_values[task]
    uncert = uncertainties[task]
    
    # Scatter plot with uncertainty
    scatter = ax.scatter(true, pred, c=uncert, cmap='viridis', alpha=0.6, s=20)
    
    # Diagonal line
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    ax.plot(lims, lims, 'r--', alpha=0.75, linewidth=2, label='Perfect prediction')
    
    # Metrics
    r2 = r2_score(true, pred)
    corr, _ = pearsonr(true, pred)
    
    ax.set_xlabel('True Value', fontsize=12)
    ax.set_ylabel('Predicted Value', fontsize=12)
    ax.set_title(f'{task}\nR²={r2:.3f}, r={corr:.3f}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Colorbar
    plt.colorbar(scatter, ax=ax, label='Uncertainty')

plt.tight_layout()
plt.savefig('predictions_vs_true.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Embryo Ranking for Selection

In [ ]:
# Create selection index (weighted sum of traits)
# Example: milk yield (40%) + protein % (30%) + fat % (30%)
trait_weights = {
    'milk_yield': 0.4,
    'protein_pct': 0.3,
    'fat_pct': 0.3
}

# Compute selection index
selection_index = np.zeros(len(predictions[task_names[0]]))

for task, weight in trait_weights.items():
    # Standardize predictions
    pred_std = (predictions[task] - predictions[task].mean()) / predictions[task].std()
    selection_index += weight * pred_std

# Rank embryos
embryo_ranks = np.argsort(selection_index)[::-1]  # Descending order

# Show top 10 embryos
print("TOP 10 EMBRYOS FOR SELECTION")
print("=" * 80)
print(f"{'Rank':<6} {'Embryo ID':<12} {'Index':<12} {'Milk Yield':<12} {'Protein %':<12} {'Fat %'}")
print("=" * 80)

for rank in range(10):
    embryo_id = embryo_ranks[rank]
    index = selection_index[embryo_id]
    
    print(f"{rank+1:<6} {embryo_id:<12} {index:<12.3f} "
          f"{predictions['milk_yield'][embryo_id]:<12.3f} "
          f"{predictions['protein_pct'][embryo_id]:<12.3f} "
          f"{predictions['fat_pct'][embryo_id]:.3f}")

print("\n✓ Embryo selection complete")

## 12. Download Results

In [ ]:
# Save results
results = {
    'predictions': predictions,
    'true_values': true_values,
    'uncertainties': uncertainties,
    'selection_index': selection_index,
    'embryo_ranks': embryo_ranks,
    'history': history,
    'config': vars(config)
}

import pickle
with open('genomic_transformer_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("✓ Results saved to genomic_transformer_results.pkl")

# Download model and results
from google.colab import files

files.download('genomic_transformer_results.pkl')
files.download('training_curves.png')
files.download('predictions_vs_true.png')

# Download best model
!zip -r best_model.zip checkpoints/genomic_transformer/best_model.pt
files.download('best_model.zip')

print("\n✓ All files ready for download!")

## 13. Summary

In [ ]:
print("=" * 80)
print("🎉 PIPELINE COMPLETE!")
print("=" * 80)

print("\n📊 FINAL RESULTS:")
for task in task_names:
    r2 = r2_score(true_values[task], predictions[task])
    corr, _ = pearsonr(true_values[task], predictions[task])
    print(f"  {task}: R²={r2:.4f}, r={corr:.4f}")

print(f"\n📦 MODEL:")
print(f"  Parameters: {n_params:,}")
print(f"  Best epoch: {best_checkpoint['epoch']}")
print(f"  Best val loss: {best_checkpoint['best_val_loss']:.4f}")

print(f"\n💾 SAVED FILES:")
print(f"  ✓ genomic_transformer_results.pkl")
print(f"  ✓ training_curves.png")
print(f"  ✓ predictions_vs_true.png")
print(f"  ✓ best_model.zip")

print("\n🧬 NEXT STEPS:")
print("  1. Fine-tune on real cattle/dog/horse genotypes")
print("  2. Integrate with embryo biopsy data")
print("  3. Validate predictions experimentally")
print("  4. Deploy for production embryo selection")

print("\n" + "=" * 80)